# AMC 2026 – Round 2 · Notebook 0: download the data
Financial Modeling (USI) · TA session 4 · *run this once before class*

This notebook downloads everything the lecture notebook needs and saves it as **CSV files in the same folder**.

| File | Content | Source |
|---|---|---|
| `r2_adj_prices.csv` | daily adjusted prices (dividends reinvested) of the 8 SAA ETFs + 3 stocks | Yahoo Finance (`yfinance`) |
| `r2_close_prices.csv` | daily raw closing prices (to compute the number of shares) | Yahoo Finance |
| `r2_dividends.csv` | dividends per share paid by the 3 stocks | Yahoo Finance |
| `r2_rates.csv` | 1-month T-bill and 10-year Treasury yield, % per year | FRED (no API key needed) |
| `r2_fundamentals.csv` | annual income statement, balance sheet and cash flow of the 3 stocks and their competitors | Yahoo Finance |
| `r2_multiples.csv` | forward EPS, trailing EPS and forward P/E of the 3 stocks and their competitors | Yahoo Finance |



In [6]:
# !pip install yfinance        # <- run once if yfinance is not installed
import pandas as pd
import yfinance as yf

ETFS   = ["SPY", "EFA", "EEM", "IEF", "LQD", "TIP", "GLD", "DBC"]   # the SAA of Round 1 (notebook 03_SAA)
STOCKS = ["NKE", "HD", "GIS"]                                        # Nike, Home Depot, General Mills
START  = "2016-01-01"                                                # > 60 months of history

# auto_adjust=True  -> prices adjusted for dividends and splits (for returns)
# auto_adjust=False -> raw closing prices (for the number of shares, as in Round 1)
adj   = yf.download(ETFS + STOCKS, start=START, auto_adjust=True,  progress=False)["Close"]
close = yf.download(ETFS + STOCKS, start=START, auto_adjust=False, progress=False)["Close"]

adj.to_csv("r2_adj_prices.csv")
close.to_csv("r2_close_prices.csv")

print(adj.shape)            # check: about 2,700 days x 11 tickers
adj.tail()

(2721, 11)


Ticker,DBC,EEM,EFA,GIS,GLD,HD,IEF,LQD,NKE,SPY,TIP
Date,,,,,,,,,,,
2026-10-22,33.570000,61.720001,103.150002,35.180000,369.000000,327.730011,91.199997,104.279999,38.320000,763.960022,106.330002
2026-10-23,34.099998,62.009998,102.629997,34.389999,366.609985,319.829987,91.050003,103.849998,37.299999,759.969971,106.239998
2026-10-26,33.779999,61.820000,102.910004,33.700001,367.820007,319.380005,91.120003,104.019997,37.910000,765.369995,106.309998
2026-10-27,33.599998,62.520000,103.589996,33.470001,372.279999,318.500000,91.099998,104.300003,37.560001,769.659973,106.309998
2026-10-28,33.904999,63.134998,103.269997,33.215000,370.359985,320.394989,90.900002,104.235001,37.544998,770.840027,106.149902


In [7]:
# Dividends per share of the 3 stocks (one row per payment date)
divs = pd.DataFrame({t: yf.Ticker(t).dividends for t in STOCKS})
divs.index = divs.index.tz_localize(None).normalize()      # drop the time zone, keep the date
divs.index.name = "Date"
divs.to_csv("r2_dividends.csv")

# check: last 4 quarterly payments of each stock
divs.tail(8)

,NKE,HD,GIS
Date,,,
2026-03-12,NaN,2.33,NaN
2026-04-10,NaN,NaN,0.61
2026-06-01,0.41,NaN,NaN
2026-06-04,NaN,2.33,NaN
2026-07-10,NaN,NaN,0.61
2026-09-01,0.41,NaN,NaN
2026-09-03,NaN,2.33,NaN
2026-10-13,NaN,NaN,0.61


In [8]:
# Interest rates from FRED, in % per year
#   DGS1MO = 1-month Treasury bill  -> risk-free rate for monthly excess returns and the Sharpe Ratio (as in Round 1)
#   DGS10  = 10-year Treasury yield -> risk-free rate in the CAPM cost of equity (as in Class 5)
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
rates = pd.concat({s: pd.read_csv(url + s, index_col=0, parse_dates=True, na_values=".").iloc[:, 0]
                   for s in ["DGS1MO", "DGS10"]}, axis=1)
rates = rates.loc[START:].ffill()
rates.index.name = "Date"
rates.to_csv("r2_rates.csv")

rates.tail()

,DGS1MO,DGS10
Date,,
2026-10-20,4.10,5.12
2026-10-21,4.09,5.13
2026-10-22,4.11,4.96
2026-10-23,4.11,5.00
2026-10-26,4.11,5.00


### Financial statements (for the value-trap check)
For the 3 stocks **and their competitors**, download the annual income statement, balance sheet and cash flow.

In [9]:
# Competitors of each stock: used for the relative valuation (§4) and the value-trap check (Checkpoint 2)
PEERS = {"NKE": ["LULU", "DECK", "ADDYY"],    # sportswear (ADDYY = Adidas ADR; its statements are in EUR, ratios are not affected)
         "HD":  ["LOW", "FND", "TSCO"],       # home-improvement retail
         "GIS": ["KHC", "CAG", "CPB"]}        # packaged food
ALL = STOCKS + [p for peers in PEERS.values() for p in peers]

# Annual financial statements of the 3 stocks and their competitors (for the value-trap check of Checkpoint 2)
rows = []
for t in ALL:
    tk = yf.Ticker(t)
    st = pd.concat([tk.income_stmt, tk.balance_sheet, tk.cashflow])   # rows = items, columns = fiscal years
    st = st[~st.index.duplicated()].T                                  # one row per fiscal year
    st["Ticker"] = t
    rows.append(st)
fund = pd.concat(rows)
fund.index.name = "Date"
fund = fund.reset_index().set_index(["Ticker", "Date"]).sort_index()
fund.to_csv("r2_fundamentals.csv")

# check: the items used in the lecture notebook must be there, for every firm
needed = ["Total Revenue", "Gross Profit", "Net Income", "Stockholders Equity", "Free Cash Flow",
          "Total Debt", "Cash Dividends Paid", "Repurchase Of Capital Stock"]

fund.loc[STOCKS, needed].round(-6)


Total Revenue  Gross Profit    Net Income  \
Ticker Date                                                    
NKE    2022-05-31            NaN           NaN           NaN   
       2023-05-31   5.121700e+10  2.229200e+10  5.070000e+09   
       2024-05-31   5.136200e+10  2.288700e+10  5.700000e+09   
       2025-05-31   4.630900e+10  1.979000e+10  3.219000e+09   
       2026-05-31   4.639800e+10  1.991100e+10  3.108000e+09   
HD     2022-01-31            NaN           NaN           NaN   
       2023-01-31   1.574030e+11  5.277800e+10  1.710500e+10   
       2024-01-31   1.526690e+11  5.096000e+10  1.514300e+10   
       2025-01-31   1.595140e+11  5.330800e+10  1.480600e+10   
       2026-01-31   1.646830e+11  5.486500e+10  1.415600e+10   
GIS    2022-05-31            NaN           NaN           NaN   
       2023-05-31   2.009400e+10  6.546000e+09  2.594000e+09   
       2024-05-31   1.985700e+10  6.932000e+09  2.497000e+09   
       2025-05-31   1.948700e+10  6.733000e+09  2.295000e+09   
       2026-05-31   1.842500e+10  6.196000e+09 -8.800000e+07   

                   Stockholders Equity  Free Cash Flow    Total Debt  \
Ticker Date                                                            
NKE    2022-05-31                  NaN             NaN           NaN   
       2023-05-31         1.400400e+10    4.872000e+09  1.214400e+10   
       2024-05-31         1.443000e+10    6.617000e+09  1.195200e+10   
       2025-05-31         1.321300e+10    3.268000e+09  1.101300e+10   
       2026-05-31         1.486500e+10    2.184000e+09  1.103300e+10   
HD     2022-01-31                  NaN             NaN           NaN   
       2023-01-31         1.562000e+09    1.149600e+10  5.036400e+10   
       2024-01-31         1.044000e+09    1.794600e+10  5.224300e+10   
       2025-01-31         6.640000e+09    1.632500e+10  6.229000e+10   
       2026-01-31         1.281300e+10    1.264600e+10  6.535000e+10   
GIS    2022-05-31                  NaN             NaN           NaN   
       2023-05-31         1.045000e+10    2.089000e+09  1.206500e+10   
       2024-05-31         9.397000e+09    2.528000e+09  1.331500e+10   
       2025-05-31         9.199000e+09    2.293000e+09  1.529700e+10   
       2026-05-31         7.368000e+09    1.626000e+09  1.394000e+10   

                   Cash Dividends Paid  Repurchase Of Capital Stock  
Ticker Date                                                          
NKE    2022-05-31                  NaN                          NaN  
       2023-05-31        -2.012000e+09                -5.480000e+09  
       2024-05-31        -2.169000e+09                -4.250000e+09  
       2025-05-31        -2.300000e+09                -2.985000e+09  
       2026-05-31        -2.407000e+09                -1.460000e+08  
HD     2022-01-31                  NaN                          NaN  
       2023-01-31        -7.789000e+09                -6.696000e+09  
       2024-01-31        -8.383000e+09                -7.951000e+09  
       2025-01-31        -8.929000e+09                -6.490000e+08  
       2026-01-31        -9.152000e+09                 0.000000e+00  
GIS    2022-05-31                  NaN                          NaN  
       2023-05-31        -1.288000e+09                -1.404000e+09  
       2024-05-31        -1.363000e+09                -2.002000e+09  
       2025-05-31        -1.339000e+09                -1.203000e+09  
       2026-05-31        -1.315000e+09                -5.000000e+08

### Multiples (for the relative valuation and the value-trap check)
For our 3 stocks and their competitors, download:
- **forward EPS** (analysts' forecast for the next 12 months) and **forward P/E**;
- **trailing EPS** (last 12 months) → forward vs trailing: do analysts expect earnings to fall? (Checkpoint 2).



In [10]:

rows = []
for s, peers in PEERS.items():
    for t, role in [(s, "stock")] + [(p, "peer") for p in peers]:
        info = yf.Ticker(t).info
        rows.append({"ticker": t, "group": s, "role": role, "forwardEps": info.get("forwardEps"),
                     "trailingEps": info.get("trailingEps"), "forwardPE": info.get("forwardPE")})
mult = pd.DataFrame(rows).set_index("ticker")
mult.to_csv("r2_multiples.csv")

print("missing values:", int(mult[["forwardEps", "trailingEps", "forwardPE"]].isna().sum().sum()))   # check: should be 0
mult


missing values: 0


,group,role,forwardEps,trailingEps,forwardPE
ticker,,,,,
NKE,NKE,stock,1.55504,2.08,24.144072
LULU,NKE,peer,8.64582,12.15,12.285128
DECK,NKE,peer,8.17759,7.55,9.771828
ADDYY,NKE,peer,6.18702,4.55,14.215244
HD,HD,stock,16.02480,14.29,19.993698
LOW,HD,peer,12.98594,11.83,15.361999
FND,HD,peer,2.13805,2.14,22.600030
TSCO,HD,peer,2.06686,1.86,15.540482
GIS,GIS,stock,3.15226,-1.64,10.536884
